In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

from tokamark.tasks import get_task_config, get_task_metadata
from tokamark.data import (
    initialize_MAST_dataset,
    initialize_TokaMark_dataset,
)


In [2]:
# ============================================================
# 1. Configuration
# ============================================================

TASK_NAME = "task_3-1"
SHOT_IDS = [27938]

ZARR_PATH = "/mnt/data2/datasets/tokamark/data"

BATCH_SIZE = 32

In [3]:
# ============================================================
# 2. Load the TokaMark task configuration
# ============================================================

task_config = get_task_config(TASK_NAME)

print("Task:", TASK_NAME)
print("Task type:", task_config["task_type"])
print(
    "Input length:",
    task_config["task_window_segmenter"]["input_length"],
)
print(
    "Output length:",
    task_config["task_window_segmenter"]["output_length"],
)



Task: task_3-1
Task type: markovian
Input length: 0.005
Output length: 0.05


In [4]:

# ============================================================
# 3. Initialize the underlying MAST dataset
# ============================================================

mast_dataset = initialize_MAST_dataset(
    config_task=task_config,
    shots_list=SHOT_IDS,
    local_flag=True,
    use_std_scaling=False,
    remove_outliers=False,
    remove_bad_efit_rating=False,
    store_manager_settings={
        "base_local_zarr_path": ZARR_PATH,
    },
)

print("\nNumber of loaded shots:", len(mast_dataset))

print(
    "Signals in first shot:",
    {
        name: signal["values"].shape
        for name, signal in mast_dataset[0].items()
    },
)



[INFO] Local Zarr database identified under `/mnt/data2/datasets/tokamark/data` with 11573 Zarr files in it.

Number of loaded shots: 1
Signals in first shot: {'thomson_scattering-t_e': (120, 140), 'thomson_scattering-n_e': (120, 140), 'pulse_schedule-i_plasma': (1, 2789), 'pulse_schedule-n_e_line': (1, 2789), 'summary-power_nbi': (1, 2789), 'gas_injection-total_injected': (1, 2789)}


In [ ]:

# ============================================================
# 3. Initialize the underlying MAST dataset
# ============================================================

mast_dataset = initialize_MAST_dataset(
    config_task=task_config,
    shots_list=SHOT_IDS,
    local_flag=True,
    use_std_scaling=False,
    remove_outliers=False,
    remove_bad_efit_rating=False,
    store_manager_settings={
        "base_local_zarr_path": ZARR_PATH,
    },
)

print("\nNumber of loaded shots:", len(mast_dataset))

print(
    "Signals in first shot:",
    {
        name: signal["values"].shape
        for name, signal in mast_dataset[0].items()
    },
)



In [5]:

# ============================================================
# 4. Apply TokaMark sliding-window segmentation
# ============================================================

tokamark_dataset = initialize_TokaMark_dataset(
    dataset=mast_dataset,
    task_metadata=get_task_metadata(task_config),
    config_metadata=task_config,
    shuffle_windows=False,
)



In [6]:

# ============================================================
# 5. PyTorch Dataset wrapper
# ============================================================

class TokaMarkTorchDataset(Dataset):
    """
    Convert a TokaMark iterable dataset into a PyTorch map-style Dataset.

    Each item contains:
        - shot_id
        - window_index
        - t_cut
        - input diagnostic signals
        - actuator signals
        - target signals
    """

    def __init__(self, tokamark_dataset):
        # TokaMarkDataset is iterable but does not implement __len__.
        # Materializing the windows allows indexing and shuffling.
        self.windows = list(tokamark_dataset)

        if not self.windows:
            raise ValueError("The TokaMark dataset contains no windows.")

        print("Number of materialized windows:", len(self.windows))

    def __len__(self):
        return len(self.windows)

    @staticmethod
    def _to_tensor(values):
        return torch.as_tensor(
            values,
            dtype=torch.float32,
        )

    def __getitem__(self, index):
        window = self.windows[index]

        inputs = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["input"].items()
        }

        actuators = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["actuator"].items()
        }

        targets = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["output"].items()
        }

        return {
            "shot_id": torch.tensor(
                window["shot_id"],
                dtype=torch.long,
            ),
            "window_index": torch.tensor(
                window["window_index"],
                dtype=torch.long,
            ),
            "t_cut": torch.tensor(
                window["t_cut"],
                dtype=torch.float32,
            ),
            "input": inputs,
            "actuator": actuators,
            "target": targets,
        }



In [7]:

# ============================================================
# 6. Create the PyTorch Dataset
# ============================================================

torch_dataset = TokaMarkTorchDataset(tokamark_dataset)

print("PyTorch dataset length:", len(torch_dataset))



Number of materialized windows: 571
PyTorch dataset length: 571


In [8]:

# ============================================================
# 7. Inspect one sample
# ============================================================

sample = torch_dataset[0]

print("\nSingle sample")
print("Shot ID:", sample["shot_id"])
print("Window index:", sample["window_index"])
print("Cut time:", sample["t_cut"])

print("\nInput shapes:")
for name, tensor in sample["input"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nActuator shapes:")
for name, tensor in sample["actuator"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nTarget shapes:")
for name, tensor in sample["target"].items():
    print(f"  {name}: {tuple(tensor.shape)}")



Single sample
Shot ID: tensor(27938)
Window index: tensor(0)
Cut time: tensor(-0.1169)

Input shapes:
  thomson_scattering-t_e: (120, 1)
  thomson_scattering-n_e: (120, 1)

Actuator shapes:
  pulse_schedule-i_plasma: (220,)
  pulse_schedule-n_e_line: (220,)
  summary-power_nbi: (220,)
  gas_injection-total_injected: (220,)

Target shapes:
  thomson_scattering-t_e: (120, 10)
  thomson_scattering-n_e: (120, 10)


In [9]:


# ============================================================
# 8. Create the DataLoader
# ============================================================

generator = torch.Generator()
generator.manual_seed(42)

data_loader = DataLoader(
    dataset=torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    generator=generator,
)


In [10]:


# ============================================================
# 9. Inspect one batch
# ============================================================

batch = next(iter(data_loader))

print("\nBatch metadata")
print("Shot IDs:", batch["shot_id"].shape)
print("Window indices:", batch["window_index"].shape)
print("Cut times:", batch["t_cut"].shape)

print("\nBatched input shapes:")
for name, tensor in batch["input"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nBatched actuator shapes:")
for name, tensor in batch["actuator"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nBatched target shapes:")
for name, tensor in batch["target"].items():
    print(f"  {name}: {tuple(tensor.shape)}")


Batch metadata
Shot IDs: torch.Size([32])
Window indices: torch.Size([32])
Cut times: torch.Size([32])

Batched input shapes:
  thomson_scattering-t_e: (32, 120, 1)
  thomson_scattering-n_e: (32, 120, 1)

Batched actuator shapes:
  pulse_schedule-i_plasma: (32, 220)
  pulse_schedule-n_e_line: (32, 220)
  summary-power_nbi: (32, 220)
  gas_injection-total_injected: (32, 220)

Batched target shapes:
  thomson_scattering-t_e: (32, 120, 10)
  thomson_scattering-n_e: (32, 120, 10)


In [16]:
import pandas as pd

from tokamark.tasks import get_task_config, get_task_metadata
from tokamark.data import (
    initialize_MAST_dataset,
    initialize_TokaMark_dataset,
)


# ============================================================
# Configuration
# ============================================================

SHOT_IDS = [27938]
ZARR_PATH = "/mnt/data2/datasets/tokamark/data"

TASK_NAMES = [
    "task_1-1",
    "task_1-2",
    "task_1-3",
    "task_2-1",
    "task_2-2",
    "task_2-3",
    "task_3-1",
    "task_3-2",
    "task_3-3",
    "task_4-1",
    "task_4-2",
    "task_4-3",
    "task_4-4",
    "task_4-5",
]


def format_signal_name(name):
    """Convert tuple signal names into readable strings."""
    if isinstance(name, tuple):
        return "/".join(map(str, name))

    return str(name)


def get_shape(values):
    """Return an object's shape as a tuple."""
    return tuple(values.shape)


# One row per task
task_records = []

# One row per signal
shape_records = []


# ============================================================
# Inspect every task
# ============================================================

for task_name in TASK_NAMES:
    print(f"\n{'=' * 70}")
    print(f"Inspecting {task_name}")
    print("=" * 70)

    try:
        task_config = get_task_config(task_name)
        segmenter_config = task_config["task_window_segmenter"]

        # Each task must initialize its own MAST dataset because
        # every task requests different signals.
        mast_dataset = initialize_MAST_dataset(
            config_task=task_config,
            shots_list=SHOT_IDS,
            local_flag=True,
            use_std_scaling=False,
            remove_outliers=False,
            remove_bad_efit_rating=False,
            store_manager_settings={
                "base_local_zarr_path": ZARR_PATH,
            },
        )

        number_of_loaded_shots = len(mast_dataset)

        if number_of_loaded_shots == 0:
            task_records.append(
                {
                    "task": task_name,
                    "task_type": task_config["task_type"],
                    "input_length": segmenter_config["input_length"],
                    "output_length": segmenter_config["output_length"],
                    "loaded_shots": 0,
                    "status": "No compatible shot",
                    "error": "",
                }
            )

            print("No compatible shot was loaded.")
            continue

        tokamark_dataset = initialize_TokaMark_dataset(
            dataset=mast_dataset,
            task_metadata=get_task_metadata(task_config),
            config_metadata=task_config,
            shuffle_windows=False,
        )

        # Only retrieve the first window. There is no need to
        # materialize every window merely to inspect the shapes.
        first_window = next(iter(tokamark_dataset), None)

        if first_window is None:
            task_records.append(
                {
                    "task": task_name,
                    "task_type": task_config["task_type"],
                    "input_length": segmenter_config["input_length"],
                    "output_length": segmenter_config["output_length"],
                    "loaded_shots": number_of_loaded_shots,
                    "status": "No valid windows",
                    "error": "",
                }
            )

            print("The shot loaded, but no valid windows were generated.")
            continue

        task_records.append(
            {
                "task": task_name,
                "task_type": task_config["task_type"],
                "input_length": segmenter_config["input_length"],
                "output_length": segmenter_config["output_length"],
                "loaded_shots": number_of_loaded_shots,
                "status": "OK",
                "error": "",
            }
        )

        print("Shot ID:", first_window["shot_id"])
        print("Window index:", first_window["window_index"])
        print("Cut time:", first_window["t_cut"])

        for section_name in ["input", "actuator", "output"]:
            signals = first_window.get(section_name, {})

            print(f"\n{section_name.capitalize()}:")

            if not signals:
                print("  No signals")
                continue

            for signal_name, signal in signals.items():
                readable_name = format_signal_name(signal_name)
                shape = get_shape(signal["values"])

                print(f"  {readable_name}: {shape}")

                shape_records.append(
                    {
                        "task": task_name,
                        "task_type": task_config["task_type"],
                        "section": section_name,
                        "signal": readable_name,
                        "shape": shape,
                        "dimensions": len(shape),
                        "number_of_values": signal["values"].size,
                    }
                )

    except Exception as error:
        print(f"Failed: {type(error).__name__}: {error}")

        task_records.append(
            {
                "task": task_name,
                "task_type": None,
                "input_length": None,
                "output_length": None,
                "loaded_shots": None,
                "status": "Failed",
                "error": f"{type(error).__name__}: {error}",
            }
        )


# ============================================================
# Results
# ============================================================

task_summary_df = pd.DataFrame(task_records)
signal_shapes_df = pd.DataFrame(shape_records)

print("\n\nTask compatibility summary")
display(task_summary_df)

print("\nSignal shapes")
display(signal_shapes_df)

OUTPUT_CSV = "tokamark_signal_shapes_shot_27938.csv"

# Convert tuple shapes such as (5, 100) into CSV-friendly strings.
signal_shapes_to_save = signal_shapes_df.copy()
signal_shapes_to_save["shape"] = signal_shapes_to_save["shape"].apply(
    lambda shape: "x".join(map(str, shape))
)

signal_shapes_to_save.to_csv(
    OUTPUT_CSV,
    index=False,
)

print(f"Saved signal shapes to: {OUTPUT_CSV}")


Inspecting task_1-1
[INFO] Local Zarr database identified under `/mnt/data2/datasets/tokamark/data` with 11573 Zarr files in it.
Shot ID: 27938
Window index: 0
Cut time: -0.11690032005310058

Input:
  magnetics-flux_loop_flux: (15, 25)
  magnetics-b_field_pol_probe_ccbv_field: (40, 25)
  magnetics-b_field_pol_probe_obr_field: (18, 25)
  magnetics-b_field_pol_probe_obv_field: (18, 25)
  magnetics-b_field_tor_probe_saddle_voltage: (12, 250)
  pf_active-coil_current: (10, 20)
  pf_active-solenoid_current: (20,)
  summary-ip: (20,)

Actuator:
  No signals

Output:
  equilibrium-elongation: (1,)
  equilibrium-elongation_axis: (1,)
  equilibrium-triangularity_upper: (1,)
  equilibrium-triangularity_lower: (1,)
  equilibrium-x_point_r: (2, 1)
  equilibrium-x_point_z: (2, 1)
  equilibrium-minor_radius: (1,)
  equilibrium-magnetic_axis_r: (1,)
  equilibrium-magnetic_axis_z: (1,)
  equilibrium-q95: (1,)
  equilibrium-beta_tor: (1,)
  equilibrium-beta_pol: (1,)
  equilibrium-beta_normal: (1,)
  

,task,task_type,input_length,output_length,loaded_shots,status,error
0,task_1-1,markovian,0.005,0.005,1,OK,
1,task_1-2,markovian,0.005,0.005,1,OK,
2,task_1-3,markovian,0.005,0.005,1,OK,
3,task_2-1,markovian,0.005,0.025,1,OK,
4,task_2-2,markovian,0.005,0.025,1,OK,
5,task_2-3,markovian,0.005,0.025,1,OK,
6,task_3-1,markovian,0.005,0.050,1,OK,
7,task_3-2,markovian,0.005,0.050,1,OK,
8,task_3-3,non_markovian,0.150,0.005,1,OK,
9,task_4-1,non_markovian,0.150,0.100,1,OK,



Signal shapes


,task,task_type,section,signal,shape,dimensions,number_of_values
0,task_1-1,markovian,input,magnetics-flux_loop_flux,"(15, 25)",2,375
1,task_1-1,markovian,input,magnetics-b_field_pol_probe_ccbv_field,"(40, 25)",2,1000
2,task_1-1,markovian,input,magnetics-b_field_pol_probe_obr_field,"(18, 25)",2,450
3,task_1-1,markovian,input,magnetics-b_field_pol_probe_obv_field,"(18, 25)",2,450
4,task_1-1,markovian,input,magnetics-b_field_tor_probe_saddle_voltage,"(12, 250)",2,3000
...,...,...,...,...,...,...,...
226,task_4-5,non_markovian,actuator,pulse_schedule-n_e_line,"(1000,)",1,1000
227,task_4-5,non_markovian,actuator,summary-power_nbi,"(1000,)",1,1000
228,task_4-5,non_markovian,actuator,gas_injection-total_injected,"(1000,)",1,1000
229,task_4-5,non_markovian,output,magnetics-b_field_tor_probe_cc_field,"(3, 256, 98)",3,75264


Saved signal shapes to: tokamark_signal_shapes_shot_27938.csv


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

from tokamark.tasks import get_task_config, get_task_metadata
from tokamark.data import (
    initialize_MAST_dataset,
    initialize_TokaMark_dataset,
)


# ============================================================
# 1. Configuration
# ============================================================

TASK_NAME = "task_3-1"
SHOT_IDS = [27938]

ZARR_PATH = "/mnt/data2/datasets/tokamark/data"

BATCH_SIZE = 32
# ============================================================
# 2. Load the TokaMark task configuration
# ============================================================

task_config = get_task_config(TASK_NAME)

print("Task:", TASK_NAME)
print("Task type:", task_config["task_type"])
print(
    "Input length:",
    task_config["task_window_segmenter"]["input_length"],
)
print(
    "Output length:",
    task_config["task_window_segmenter"]["output_length"],
)




# ============================================================
# 3. Initialize the underlying MAST dataset
# ============================================================

mast_dataset = initialize_MAST_dataset(
    config_task=task_config,
    shots_list=SHOT_IDS,
    local_flag=True,
    use_std_scaling=False,
    remove_outliers=False,
    remove_bad_efit_rating=False,
    store_manager_settings={
        "base_local_zarr_path": ZARR_PATH,
    },
)

print("\nNumber of loaded shots:", len(mast_dataset))

print(
    "Signals in first shot:",
    {
        name: signal["values"].shape
        for name, signal in mast_dataset[0].items()
    },
)


# ============================================================
# 4. Apply TokaMark sliding-window segmentation
# ============================================================

tokamark_dataset = initialize_TokaMark_dataset(
    dataset=mast_dataset,
    task_metadata=get_task_metadata(task_config),
    config_metadata=task_config,
    shuffle_windows=False,
)


# ============================================================
# 5. PyTorch Dataset wrapper
# ============================================================

class TokaMarkTorchDataset(Dataset):
    """
    Convert a TokaMark iterable dataset into a PyTorch map-style Dataset.

    Each item contains:
        - shot_id
        - window_index
        - t_cut
        - input diagnostic signals
        - actuator signals
        - target signals
    """

    def __init__(self, tokamark_dataset):
        # TokaMarkDataset is iterable but does not implement __len__.
        # Materializing the windows allows indexing and shuffling.
        self.windows = list(tokamark_dataset)

        if not self.windows:
            raise ValueError("The TokaMark dataset contains no windows.")

        print("Number of materialized windows:", len(self.windows))

    def __len__(self):
        return len(self.windows)

    @staticmethod
    def _to_tensor(values):
        return torch.as_tensor(
            values,
            dtype=torch.float32,
        )

    def __getitem__(self, index):
        window = self.windows[index]

        inputs = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["input"].items()
        }

        actuators = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["actuator"].items()
        }

        targets = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["output"].items()
        }

        return {
            "shot_id": torch.tensor(
                window["shot_id"],
                dtype=torch.long,
            ),
            "window_index": torch.tensor(
                window["window_index"],
                dtype=torch.long,
            ),
            "t_cut": torch.tensor(
                window["t_cut"],
                dtype=torch.float32,
            ),
            "input": inputs,
            "actuator": actuators,
            "target": targets,
        }


# ============================================================
# 6. Create the PyTorch Dataset
# ============================================================

torch_dataset = TokaMarkTorchDataset(tokamark_dataset)

print("PyTorch dataset length:", len(torch_dataset))


# ============================================================
# 7. Inspect one sample
# ============================================================

sample = torch_dataset[0]

print("\nSingle sample")
print("Shot ID:", sample["shot_id"])
print("Window index:", sample["window_index"])
print("Cut time:", sample["t_cut"])

print("\nInput shapes:")
for name, tensor in sample["input"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nActuator shapes:")
for name, tensor in sample["actuator"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nTarget shapes:")
for name, tensor in sample["target"].items():
    print(f"  {name}: {tuple(tensor.shape)}")


# ============================================================
# 8. Create the DataLoader
# ============================================================

generator = torch.Generator()
generator.manual_seed(42)

data_loader = DataLoader(
    dataset=torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    generator=generator,
)


# ============================================================
# 9. Inspect one batch
# ============================================================

batch = next(iter(data_loader))

print("\nBatch metadata")
print("Shot IDs:", batch["shot_id"].shape)
print("Window indices:", batch["window_index"].shape)
print("Cut times:", batch["t_cut"].shape)

print("\nBatched input shapes:")
for name, tensor in batch["input"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nBatched actuator shapes:")
for name, tensor in batch["actuator"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

print("\nBatched target shapes:")
for name, tensor in batch["target"].items():
    print(f"  {name}: {tuple(tensor.shape)}")

In [17]:
import gc
import torch
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from tokamark.tasks import get_task_config, get_task_metadata
from tokamark.data import (
    initialize_MAST_dataset,
    initialize_TokaMark_dataset,
)


# ============================================================
# 1. Configuration
# ============================================================

SHOT_IDS = [27938]
ZARR_PATH = "/mnt/data2/datasets/tokamark/data"
BATCH_SIZE = 32

TASK_NAMES = [
    "task_1-1",
    "task_1-2",
    "task_1-3",
    "task_2-1",
    "task_2-2",
    "task_2-3",
    "task_3-1",
    "task_3-2",
    "task_3-3",
    "task_4-1",
    "task_4-2",
    "task_4-3",
    "task_4-4",
    "task_4-5",
]

OUTPUT_CSV = "tokamark_batch_shapes_shot_27938.csv"


# ============================================================
# 2. PyTorch Dataset wrapper
# ============================================================

class TokaMarkTorchDataset(Dataset):
    def __init__(self, tokamark_dataset):
        self.windows = list(tokamark_dataset)

        if not self.windows:
            raise ValueError(
                "The TokaMark dataset contains no valid windows."
            )

    def __len__(self):
        return len(self.windows)

    @staticmethod
    def _to_tensor(values):
        return torch.as_tensor(
            values,
            dtype=torch.float32,
        )

    def __getitem__(self, index):
        window = self.windows[index]

        inputs = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["input"].items()
        }

        actuators = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["actuator"].items()
        }

        targets = {
            name: self._to_tensor(signal["values"])
            for name, signal in window["output"].items()
        }

        return {
            "shot_id": torch.tensor(
                window["shot_id"],
                dtype=torch.long,
            ),
            "window_index": torch.tensor(
                window["window_index"],
                dtype=torch.long,
            ),
            "t_cut": torch.tensor(
                window["t_cut"],
                dtype=torch.float32,
            ),
            "input": inputs,
            "actuator": actuators,
            "target": targets,
        }


# ============================================================
# 3. Helper functions
# ============================================================

def format_signal_name(name):
    if isinstance(name, (tuple, list)):
        return "/".join(map(str, name))

    return str(name)


def shape_to_string(shape):
    return "x".join(map(str, shape))


def add_batch_shapes(
    records,
    task_name,
    task_type,
    section_name,
    signals,
    number_of_windows,
):
    for signal_name, tensor in signals.items():
        batch_shape = tuple(tensor.shape)
        sample_shape = tuple(tensor.shape[1:])

        records.append(
            {
                "task": task_name,
                "task_type": task_type,
                "section": section_name,
                "signal": format_signal_name(signal_name),
                "number_of_windows": number_of_windows,
                "requested_batch_size": BATCH_SIZE,
                "actual_batch_size": batch_shape[0],
                "batch_shape": shape_to_string(batch_shape),
                "sample_shape": shape_to_string(sample_shape),
                "dimensions_with_batch": tensor.ndim,
                "dtype": str(tensor.dtype),
            }
        )


# ============================================================
# 4. Create one DataLoader per task
# ============================================================

shape_records = []
task_summary_records = []

for task_name in TASK_NAMES:
    print(f"\n{'=' * 70}")
    print(f"Task: {task_name}")
    print("=" * 70)

    try:
        task_config = get_task_config(task_name)
        task_type = task_config["task_type"]
        segmenter_config = task_config["task_window_segmenter"]

        # ----------------------------------------------------
        # Initialize task-specific MAST dataset
        # ----------------------------------------------------

        mast_dataset = initialize_MAST_dataset(
            config_task=task_config,
            shots_list=SHOT_IDS,
            local_flag=True,
            use_std_scaling=False,
            remove_outliers=False,
            remove_bad_efit_rating=False,
            store_manager_settings={
                "base_local_zarr_path": ZARR_PATH,
            },
        )

        if len(mast_dataset) == 0:
            raise ValueError(
                f"Shot {SHOT_IDS} is not compatible with {task_name}."
            )

        # ----------------------------------------------------
        # Create task-specific TokaMark dataset
        # ----------------------------------------------------

        tokamark_dataset = initialize_TokaMark_dataset(
            dataset=mast_dataset,
            task_metadata=get_task_metadata(task_config),
            config_metadata=task_config,
            shuffle_windows=False,
        )

        # ----------------------------------------------------
        # Create PyTorch Dataset and DataLoader
        # ----------------------------------------------------

        torch_dataset = TokaMarkTorchDataset(
            tokamark_dataset
        )

        number_of_windows = len(torch_dataset)

        generator = torch.Generator()
        generator.manual_seed(42)

        data_loader = DataLoader(
            dataset=torch_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
            generator=generator,
        )

        # Retrieve an actual PyTorch batch
        batch = next(iter(data_loader))

        actual_batch_size = batch["shot_id"].shape[0]

        print("Task type:", task_type)
        print(
            "Input length:",
            segmenter_config["input_length"],
        )
        print(
            "Output length:",
            segmenter_config["output_length"],
        )
        print("Number of windows:", number_of_windows)
        print("Actual batch size:", actual_batch_size)

        # ----------------------------------------------------
        # Print and record input shapes
        # ----------------------------------------------------

        print("\nBatched input shapes:")

        for name, tensor in batch["input"].items():
            print(
                f"  {format_signal_name(name)}: "
                f"{tuple(tensor.shape)}"
            )

        add_batch_shapes(
            records=shape_records,
            task_name=task_name,
            task_type=task_type,
            section_name="input",
            signals=batch["input"],
            number_of_windows=number_of_windows,
        )

        # ----------------------------------------------------
        # Print and record actuator shapes
        # ----------------------------------------------------

        print("\nBatched actuator shapes:")

        if not batch["actuator"]:
            print("  No actuator signals")

        for name, tensor in batch["actuator"].items():
            print(
                f"  {format_signal_name(name)}: "
                f"{tuple(tensor.shape)}"
            )

        add_batch_shapes(
            records=shape_records,
            task_name=task_name,
            task_type=task_type,
            section_name="actuator",
            signals=batch["actuator"],
            number_of_windows=number_of_windows,
        )

        # ----------------------------------------------------
        # Print and record target shapes
        # ----------------------------------------------------

        print("\nBatched target shapes:")

        for name, tensor in batch["target"].items():
            print(
                f"  {format_signal_name(name)}: "
                f"{tuple(tensor.shape)}"
            )

        add_batch_shapes(
            records=shape_records,
            task_name=task_name,
            task_type=task_type,
            section_name="target",
            signals=batch["target"],
            number_of_windows=number_of_windows,
        )

        task_summary_records.append(
            {
                "task": task_name,
                "task_type": task_type,
                "input_length": segmenter_config["input_length"],
                "output_length": segmenter_config["output_length"],
                "number_of_windows": number_of_windows,
                "actual_batch_size": actual_batch_size,
                "status": "OK",
                "error": "",
            }
        )

    except Exception as error:
        print(
            f"Failed: {type(error).__name__}: {error}"
        )

        task_summary_records.append(
            {
                "task": task_name,
                "task_type": None,
                "input_length": None,
                "output_length": None,
                "number_of_windows": 0,
                "actual_batch_size": 0,
                "status": "Failed",
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            }
        )

    finally:
        # Release the materialized windows before loading
        # the next task.
        for variable_name in [
            "batch",
            "data_loader",
            "torch_dataset",
            "tokamark_dataset",
            "mast_dataset",
        ]:
            if variable_name in locals():
                del locals()[variable_name]

        gc.collect()


# ============================================================
# 5. Save results
# ============================================================

batch_shapes_df = pd.DataFrame(shape_records)
task_summary_df = pd.DataFrame(task_summary_records)

batch_shapes_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

task_summary_df.to_csv(
    "tokamark_batch_task_summary_shot_27938.csv",
    index=False,
)

print(f"\nSaved batch shapes to: {OUTPUT_CSV}")
print(
    "Saved task summary to: "
    "tokamark_batch_task_summary_shot_27938.csv"
)

display(task_summary_df)
display(batch_shapes_df)


Task: task_1-1
[INFO] Local Zarr database identified under `/mnt/data2/datasets/tokamark/data` with 11573 Zarr files in it.
Task type: markovian
Input length: 0.005
Output length: 0.005
Number of windows: 616
Actual batch size: 32

Batched input shapes:
  magnetics-flux_loop_flux: (32, 15, 25)
  magnetics-b_field_pol_probe_ccbv_field: (32, 40, 25)
  magnetics-b_field_pol_probe_obr_field: (32, 18, 25)
  magnetics-b_field_pol_probe_obv_field: (32, 18, 25)
  magnetics-b_field_tor_probe_saddle_voltage: (32, 12, 250)
  pf_active-coil_current: (32, 10, 20)
  pf_active-solenoid_current: (32, 20)
  summary-ip: (32, 20)

Batched actuator shapes:
  No actuator signals

Batched target shapes:
  equilibrium-elongation: (32, 1)
  equilibrium-elongation_axis: (32, 1)
  equilibrium-triangularity_upper: (32, 1)
  equilibrium-triangularity_lower: (32, 1)
  equilibrium-x_point_r: (32, 2, 1)
  equilibrium-x_point_z: (32, 2, 1)
  equilibrium-minor_radius: (32, 1)
  equilibrium-magnetic_axis_r: (32, 1)
  

,task,task_type,input_length,output_length,number_of_windows,actual_batch_size,status,error
0,task_1-1,markovian,0.005,0.005,616,32,OK,
1,task_1-2,markovian,0.005,0.005,616,32,OK,
2,task_1-3,markovian,0.005,0.005,621,32,OK,
3,task_2-1,markovian,0.005,0.025,668,32,OK,
4,task_2-2,markovian,0.005,0.025,591,32,OK,
5,task_2-3,markovian,0.005,0.025,596,32,OK,
6,task_3-1,markovian,0.005,0.050,571,32,OK,
7,task_3-2,markovian,0.005,0.050,643,32,OK,
8,task_3-3,NaN,NaN,NaN,0,0,Failed,RuntimeError: stack expects each tensor to be ...
9,task_4-1,NaN,NaN,NaN,0,0,Failed,RuntimeError: stack expects each tensor to be ...


,task,task_type,section,signal,number_of_windows,requested_batch_size,actual_batch_size,batch_shape,sample_shape,dimensions_with_batch,dtype
0,task_1-1,markovian,input,magnetics-flux_loop_flux,616,32,32,32x15x25,15x25,3,torch.float32
1,task_1-1,markovian,input,magnetics-b_field_pol_probe_ccbv_field,616,32,32,32x40x25,40x25,3,torch.float32
2,task_1-1,markovian,input,magnetics-b_field_pol_probe_obr_field,616,32,32,32x18x25,18x25,3,torch.float32
3,task_1-1,markovian,input,magnetics-b_field_pol_probe_obv_field,616,32,32,32x18x25,18x25,3,torch.float32
4,task_1-1,markovian,input,magnetics-b_field_tor_probe_saddle_voltage,616,32,32,32x12x250,12x250,3,torch.float32
...,...,...,...,...,...,...,...,...,...,...,...
108,task_3-2,markovian,actuator,summary-power_nbi,643,32,32,32x220,220,2,torch.float32
109,task_3-2,markovian,actuator,gas_injection-total_injected,643,32,32,32x220,220,2,torch.float32
110,task_3-2,markovian,target,spectrometer_visible-filter_spectrometer_dalph...,643,32,32,32x3x2500,3x2500,3,torch.float32
111,task_3-2,markovian,target,soft_x_rays-horizontal_cam_lower,643,32,32,32x18x2500,18x2500,3,torch.float32
